<a href="https://colab.research.google.com/github/WeegorMartins/customer-decisioning-lab/blob/main/notebooks/05_economics_and_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install "ortools==9.12.4544"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 9.3 MB/s eta 0:00:00


In [2]:
import ortools
print("OR-Tools:", ortools.__version__)

OR-Tools: 9.12.4544


In [3]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from ortools.sat.python import cp_model

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
rng = np.random.default_rng(SEED)

PROJECT_DIR = Path(
    "/content/drive/MyDrive/customer-decisioning-lab"
)

SCORED_PATH = (
    PROJECT_DIR
    / "results"
    / "card_oot_scored.parquet"
)

scored = pd.read_parquet(SCORED_PATH)

print("Formato:", scored.shape)
display(scored.head())

Mounted at /content/drive
Formato: (17747, 32)


,customer_id,decision_date,tenure_months,age,spend_90d,transactions_90d,recency_days,digital_sessions_30d,merchant_diversity_90d,contacts_30d,...,p_control_for_1,uplift_action_1,p_action_2,p_control_for_2,uplift_action_2,p_action_3,p_control_for_3,uplift_action_3,best_action_by_uplift,best_uplift
0,16,2026-03-01,70,60,445.61,6,7,5,14,0,...,0.064062,0.127066,0.071178,0.064062,0.007116,0.088106,0.064062,0.024043,1,0.127066
1,25,2026-03-01,39,26,2623.99,13,28,6,1,0,...,0.229233,0.326859,0.416160,0.229233,0.186927,0.247531,0.229233,0.018298,1,0.326859
2,28,2026-03-01,16,44,901.93,10,1,1,1,3,...,0.082005,-0.020698,0.113457,0.082005,0.031452,0.078952,0.082005,-0.003052,2,0.031452
3,38,2026-03-01,66,21,569.66,15,18,3,9,1,...,0.030124,0.075112,0.067624,0.030124,0.037500,0.047056,0.030124,0.016931,1,0.075112
4,50,2026-03-01,37,65,379.71,15,14,3,15,0,...,0.017736,0.064241,0.079457,0.017736,0.061722,0.198152,0.017736,0.180416,3,0.180416


# Parte B — regras econômicas

In [4]:
ACTION_NAMES = {
    0: "no_contact",
    1: "cashback",
    2: "double_points",
    3: "installment"
}

UNIT_COST = {
    0: 0.0,
    1: 12.0,
    2: 7.0,
    3: 5.0
}

CONVERSION_VALUE = 160.0
OPTOUT_PENALTY = 40.0

TOTAL_BUDGET = round(
    6.0 * len(scored),
    2
)

MAX_CONTACTS = int(
    0.45 * len(scored)
)

ACTION_CAPS = {
    1: int(0.22 * len(scored)),
    2: int(0.25 * len(scored)),
    3: int(0.18 * len(scored))
}

CONTROL_SHARE = 0.10

print("Clientes:", len(scored))
print("Orçamento:", TOTAL_BUDGET)
print("Máximo de contatos:", MAX_CONTACTS)
print("Capacidades:", ACTION_CAPS)

Clientes: 17747
Orçamento: 106482.0
Máximo de contatos: 7986
Capacidades: {1: 3904, 2: 4436, 3: 3194}


- cada conversão adicional vale R$160 no cenário;

- cashback custa R$12;

- opt-out possui penalidade econômica de R$40;

- orçamento escala com o tamanho da amostra.

# Parte C — prever risco de opt-out

In [5]:
def sigmoid(value):
    return 1 / (1 + np.exp(-value))


def predicted_optout_probability(
    data,
    action
):
    logit = (
        -5.20
        + 0.60 * data["contacts_30d"]
        + 0.75 * (
            data["complaints_180d"] > 0
        )
        + 0.45 * (action != 0)
    )

    return np.clip(
        sigmoid(logit),
        0.001,
        0.50
    )

In [6]:
for action in [0, 1, 2, 3]:
    scored[
        f"predicted_optout_action_{action}"
    ] = predicted_optout_probability(
        scored,
        action
    )

display(
    scored[[
        "customer_id",
        "predicted_optout_action_0",
        "predicted_optout_action_1"
    ]].head()
)

,customer_id,predicted_optout_action_0,predicted_optout_action_1
0,16,0.005486,0.008577
1,25,0.005486,0.008577
2,28,0.032295,0.049737
3,38,0.009952,0.015520
4,50,0.005486,0.008577


# Parte D — transformar uplift em valor

In [7]:
scored["value_action_0"] = 0.0

In [8]:
for action in [1, 2, 3]:
    scored[f"value_action_{action}"] = (
        scored[f"uplift_action_{action}"]
        * CONVERSION_VALUE
        - UNIT_COST[action]
        - scored[
            f"predicted_optout_action_{action}"
        ]
        * OPTOUT_PENALTY
    )

display(
    scored[[
        "customer_id",
        "value_action_0",
        "value_action_1",
        "value_action_2",
        "value_action_3"
    ]].head()
)

,customer_id,value_action_0,value_action_1,value_action_2,value_action_3
0,16,0.0,7.987455,-6.204546,-1.496150
1,25,0.0,39.954321,22.565247,-2.415386
2,28,0.0,-17.301094,-3.957130,-7.477856
3,38,0.0,-0.602836,-1.620834,-2.911789
4,50,0.0,-2.064493,2.532357,23.523464


# Parte E — reservar controle

In [9]:
scored["reserved_control"] = 0

control_size = int(
    CONTROL_SHARE * len(scored)
)

control_indices = rng.choice(
    scored.index.to_numpy(),
    size=control_size,
    replace=False
)

scored.loc[
    control_indices,
    "reserved_control"
] = 1

print(
    scored["reserved_control"]
    .value_counts()
)

reserved_control
0    15973
1     1774
Name: count, dtype: int64


# Parte F — elegibilidade

In [10]:
def eligible_for_action(
    data,
    action
):
    if action == 0:
        return pd.Series(
            True,
            index=data.index
        )

    eligible = (
        data["marketing_consent"].eq(1)
        & data["contacts_30d"].le(2)
        & data["reserved_control"].eq(0)
    )

    if action == 3:
        eligible = (
            eligible
            & data[
                "lifecycle_segment"
            ].ne("novo")
        )

    return eligible

In [11]:
for action in [0, 1, 2, 3]:
    scored[
        f"eligible_action_{action}"
    ] = eligible_for_action(
        scored,
        action
    )

eligibility_summary = pd.DataFrame({
    "action": [0, 1, 2, 3],
    "eligible_customers": [
        scored[
            f"eligible_action_{action}"
        ].sum()
        for action in [0, 1, 2, 3]
    ]
})

display(eligibility_summary)

,action,eligible_customers
0,0,17747
1,1,11735
2,2,11735
3,3,11426


In [12]:
assert not scored.loc[
    scored["marketing_consent"].eq(0),
    "eligible_action_1"
].any()

assert not scored.loc[
    scored["reserved_control"].eq(1),
    "eligible_action_2"
].any()

assert scored["eligible_action_0"].all()

print("REGRAS DE ELEGIBILIDADE PASSARAM")

REGRAS DE ELEGIBILIDADE PASSARAM


# Parte G — matriz longa de candidatos

In [13]:
candidate_parts = []

for action in [0, 1, 2, 3]:
    part = pd.DataFrame({
        "customer_id":
            scored["customer_id"],
        "action": action,
        "action_name":
            ACTION_NAMES[action],
        "expected_value":
            scored[f"value_action_{action}"],
        "unit_cost":
            UNIT_COST[action],
        "eligible":
            scored[
                f"eligible_action_{action}"
            ],
        "reserved_control":
            scored["reserved_control"]
    })

    candidate_parts.append(part)

candidates = pd.concat(
    candidate_parts,
    ignore_index=True
)

display(candidates.head(8))
print("Linhas:", len(candidates))

,customer_id,action,action_name,expected_value,unit_cost,eligible,reserved_control
0,16,0,no_contact,0.0,0.0,True,0
1,25,0,no_contact,0.0,0.0,True,0
2,28,0,no_contact,0.0,0.0,True,0
3,38,0,no_contact,0.0,0.0,True,0
4,50,0,no_contact,0.0,0.0,True,0
5,71,0,no_contact,0.0,0.0,True,0
6,84,0,no_contact,0.0,0.0,True,0
7,90,0,no_contact,0.0,0.0,True,0


Linhas: 70988


In [14]:
assert (
    candidates.groupby(
        "customer_id"
    ).size() == 4
).all()

assert candidates[
    candidates["action"] == 0
]["eligible"].all()

print("MATRIZ DE CANDIDATOS VÁLIDA")

MATRIZ DE CANDIDATOS VÁLIDA


# Parte H — política greedy

In [15]:
contact_candidates = candidates[
    candidates["eligible"]
    & candidates["action"].ne(0)
    & candidates["expected_value"].gt(0)
].copy()

contact_candidates[
    "value_per_cost"
] = (
    contact_candidates["expected_value"]
    / contact_candidates["unit_cost"]
)

contact_candidates = (
    contact_candidates
    .sort_values(
        [
            "value_per_cost",
            "expected_value"
        ],
        ascending=False
    )
    .drop_duplicates(
        "customer_id",
        keep="first"
    )
)

print(
    "Candidatos positivos:",
    len(contact_candidates)
)

Candidatos positivos: 9431


In [16]:
selected_rows = []
used_budget = 0.0
used_contacts = 0
used_caps = {
    1: 0,
    2: 0,
    3: 0
}

for row in contact_candidates.itertuples():
    action = int(row.action)

    if used_contacts >= MAX_CONTACTS:
        break

    if (
        used_budget + row.unit_cost
        > TOTAL_BUDGET
    ):
        continue

    if (
        used_caps[action]
        >= ACTION_CAPS[action]
    ):
        continue

    selected_rows.append({
        "customer_id": row.customer_id,
        "recommended_action": action,
        "expected_value":
            row.expected_value,
        "unit_cost": row.unit_cost
    })

    used_budget += row.unit_cost
    used_contacts += 1
    used_caps[action] += 1

greedy_selected = pd.DataFrame(
    selected_rows
)

print("Contatos:", used_contacts)
print("Custo:", used_budget)
print("Capacidades:", used_caps)

Contatos: 7986
Custo: 58041.0
Capacidades: {1: 1631, 2: 3347, 3: 3008}


In [17]:
greedy_policy = scored[[
    "customer_id",
    "reserved_control"
]].copy()

greedy_policy = greedy_policy.merge(
    greedy_selected,
    on="customer_id",
    how="left"
)

greedy_policy[
    "recommended_action"
] = (
    greedy_policy[
        "recommended_action"
    ]
    .fillna(0)
    .astype(int)
)

greedy_policy["expected_value"] = (
    greedy_policy["expected_value"]
    .fillna(0.0)
)

greedy_policy["unit_cost"] = (
    greedy_policy["unit_cost"]
    .fillna(0.0)
)

display(
    greedy_policy[
        "recommended_action"
    ].value_counts()
)

,count
recommended_action,
0,9761
2,3347
3,3008
1,1631


In [18]:
assert greedy_policy[
    "customer_id"
].is_unique

assert (
    greedy_policy["unit_cost"].sum()
    <= TOTAL_BUDGET + 1e-9
)

assert (
    greedy_policy[
        "recommended_action"
    ].ne(0).sum()
    <= MAX_CONTACTS
)

print("POLÍTICA GREEDY VÁLIDA")

POLÍTICA GREEDY VÁLIDA


# Parte I — reduzir o problema para o solver

In [19]:
SHORTLIST_CUSTOMERS = min(
    25_000,
    len(scored)
)

customer_priority = (
    scored.set_index("customer_id")[[
        "value_action_1",
        "value_action_2",
        "value_action_3"
    ]]
    .max(axis=1)
    .sort_values(ascending=False)
)

shortlist_ids = (
    customer_priority
    .head(SHORTLIST_CUSTOMERS)
    .index
)

solver_candidates = candidates[
    candidates[
        "customer_id"
    ].isin(shortlist_ids)
    & candidates["eligible"]
].reset_index(drop=True)

print(
    "Variáveis candidatas:",
    len(solver_candidates)
)

Variáveis candidatas: 52643


# Parte J — OR-Tools

In [20]:
model = cp_model.CpModel()

x = {}

for idx, row in solver_candidates.iterrows():
    x[idx] = model.NewBoolVar(
        f"x_{int(row.customer_id)}_"
        f"{int(row.action)}"
    )

print("Variáveis criadas:", len(x))

Variáveis criadas: 52643


In [21]:
for customer_id, group in (
    solver_candidates.groupby(
        "customer_id"
    )
):
    model.Add(
        sum(
            x[idx]
            for idx in group.index
        ) == 1
    )

In [22]:
BUDGET_CENTS = int(
    round(TOTAL_BUDGET * 100)
)

model.Add(
    sum(
        x[idx]
        * int(
            round(
                row.unit_cost * 100
            )
        )
        for idx, row
        in solver_candidates.iterrows()
    )
    <= BUDGET_CENTS
)

In [23]:
contact_indices = (
    solver_candidates.index[
        solver_candidates["action"] != 0
    ]
)

model.Add(
    sum(
        x[idx]
        for idx in contact_indices
    )
    <= MAX_CONTACTS
)

In [24]:
for action, cap in ACTION_CAPS.items():
    action_indices = (
        solver_candidates.index[
            solver_candidates[
                "action"
            ].eq(action)
        ]
    )

    model.Add(
        sum(
            x[idx]
            for idx in action_indices
        )
        <= cap
    )

In [25]:
model.Maximize(
    sum(
        x[idx]
        * int(
            round(
                row.expected_value * 100
            )
        )
        for idx, row
        in solver_candidates.iterrows()
    )
)

In [26]:
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 120
solver.parameters.num_search_workers = 8

start = time.time()
status = solver.Solve(model)
elapsed = time.time() - start

print("Status:", solver.StatusName(status))
print("Segundos:", round(elapsed, 1))

Status: OPTIMAL
Segundos: 111.4


In [27]:
optimized_rows = []

if status in (
    cp_model.OPTIMAL,
    cp_model.FEASIBLE
):
    for idx, row in (
        solver_candidates.iterrows()
    ):
        if solver.Value(x[idx]) == 1:
            optimized_rows.append({
                "customer_id":
                    int(row.customer_id),
                "recommended_action":
                    int(row.action),
                "expected_value":
                    float(row.expected_value),
                "unit_cost":
                    float(row.unit_cost)
            })

optimized_shortlist = pd.DataFrame(
    optimized_rows
)

display(
    optimized_shortlist[
        "recommended_action"
    ].value_counts()
)

,count
recommended_action,
0,9761
2,2919
1,2911
3,2156


In [28]:
optimized_policy = scored[[
    "customer_id",
    "reserved_control"
]].copy()

optimized_policy = optimized_policy.merge(
    optimized_shortlist,
    on="customer_id",
    how="left"
)

optimized_policy[
    "recommended_action"
] = (
    optimized_policy[
        "recommended_action"
    ]
    .fillna(0)
    .astype(int)
)

optimized_policy["expected_value"] = (
    optimized_policy["expected_value"]
    .fillna(0.0)
)

optimized_policy["unit_cost"] = (
    optimized_policy["unit_cost"]
    .fillna(0.0)
)

In [29]:
assert optimized_policy[
    "customer_id"
].is_unique

assert (
    optimized_policy[
        "unit_cost"
    ].sum()
    <= TOTAL_BUDGET + 1e-9
)

assert (
    optimized_policy[
        "recommended_action"
    ].ne(0).sum()
    <= MAX_CONTACTS
)

for action, cap in ACTION_CAPS.items():
    assert (
        optimized_policy[
            "recommended_action"
        ].eq(action).sum()
        <= cap
    )

print("POLÍTICA OTIMIZADA VÁLIDA")

POLÍTICA OTIMIZADA VÁLIDA


# Parte K — comparação prevista

In [30]:
predicted_comparison = pd.DataFrame([
    {
        "policy": "greedy",
        "predicted_value":
            greedy_policy[
                "expected_value"
            ].sum(),
        "cost":
            greedy_policy[
                "unit_cost"
            ].sum(),
        "contacts":
            greedy_policy[
                "recommended_action"
            ].ne(0).sum()
    },
    {
        "policy": "optimized",
        "predicted_value":
            optimized_policy[
                "expected_value"
            ].sum(),
        "cost":
            optimized_policy[
                "unit_cost"
            ].sum(),
        "contacts":
            optimized_policy[
                "recommended_action"
            ].ne(0).sum()
    }
])

display(predicted_comparison)

,policy,predicted_value,cost,contacts
0,greedy,144273.683742,58041.0,7986
1,optimized,150833.950275,66145.0,7986


# Parte L — avaliação DR multi-ação

In [31]:
mu = {
    0: scored[
        "p_control_for_1"
    ].to_numpy(),
    1: scored[
        "p_action_1"
    ].to_numpy(),
    2: scored[
        "p_action_2"
    ].to_numpy(),
    3: scored[
        "p_action_3"
    ].to_numpy()
}

penalty = {
    0: np.zeros(len(scored)),
    1: (
        scored[
            "predicted_optout_action_1"
        ].to_numpy()
        * OPTOUT_PENALTY
    ),
    2: (
        scored[
            "predicted_optout_action_2"
        ].to_numpy()
        * OPTOUT_PENALTY
    ),
    3: (
        scored[
            "predicted_optout_action_3"
        ].to_numpy()
        * OPTOUT_PENALTY
    )
}

In [32]:
def multi_action_dr_score(
    observed_action,
    observed_outcome,
    recommended_action,
    mu,
    unit_cost,
    penalty,
    treatment_probability,
    conversion_value
):
    observed_action = np.asarray(
        observed_action,
        dtype=int
    )
    observed_outcome = np.asarray(
        observed_outcome,
        dtype=float
    )
    recommended_action = np.asarray(
        recommended_action,
        dtype=int
    )

    n = len(observed_action)
    row_idx = np.arange(n)

    mu_matrix = np.column_stack([
        mu[action]
        for action in [0, 1, 2, 3]
    ])

    penalty_matrix = np.column_stack([
        penalty[action]
        for action in [0, 1, 2, 3]
    ])

    cost_array = np.array([
        unit_cost[action]
        for action in [0, 1, 2, 3]
    ])

    predicted_reward_matrix = (
        conversion_value * mu_matrix
        - cost_array
        - penalty_matrix
    )

    mu_recommended = (
        predicted_reward_matrix[
            row_idx,
            recommended_action
        ]
    )

    mu_observed = (
        predicted_reward_matrix[
            row_idx,
            observed_action
        ]
    )

    observed_reward = (
        conversion_value
        * observed_outcome
        - cost_array[observed_action]
        - penalty_matrix[
            row_idx,
            observed_action
        ]
    )

    matching = (
        observed_action
        == recommended_action
    )

    correction = (
        matching
        * (
            observed_reward - mu_observed
        )
        / treatment_probability
    )

    return mu_recommended + correction

In [33]:
observed_action = (
    scored["treatment"].to_numpy()
)
observed_outcome = (
    scored["converted_90d"].to_numpy()
)

control_recommendation = np.zeros(
    len(scored),
    dtype=int
)

control_score = multi_action_dr_score(
    observed_action,
    observed_outcome,
    control_recommendation,
    mu,
    UNIT_COST,
    penalty,
    treatment_probability=0.25,
    conversion_value=CONVERSION_VALUE
)

greedy_score = multi_action_dr_score(
    observed_action,
    observed_outcome,
    greedy_policy[
        "recommended_action"
    ],
    mu,
    UNIT_COST,
    penalty,
    treatment_probability=0.25,
    conversion_value=CONVERSION_VALUE
)

optimized_score = multi_action_dr_score(
    observed_action,
    observed_outcome,
    optimized_policy[
        "recommended_action"
    ],
    mu,
    UNIT_COST,
    penalty,
    treatment_probability=0.25,
    conversion_value=CONVERSION_VALUE
)

In [34]:
def bootstrap_difference(
    policy_score,
    baseline_score,
    n_boot=500,
    seed=42
):
    difference = (
        np.asarray(policy_score)
        - np.asarray(baseline_score)
    )

    rng_boot = np.random.default_rng(seed)
    estimates = np.empty(n_boot)

    for i in range(n_boot):
        idx = rng_boot.integers(
            0,
            len(difference),
            size=len(difference)
        )
        estimates[i] = difference[
            idx
        ].mean()

    return {
        "incremental_value_per_customer":
            float(difference.mean()),
        "ci_low":
            float(
                np.quantile(
                    estimates,
                    0.025
                )
            ),
        "ci_high":
            float(
                np.quantile(
                    estimates,
                    0.975
                )
            )
    }

greedy_dr = bootstrap_difference(
    greedy_score,
    control_score
)

optimized_dr = bootstrap_difference(
    optimized_score,
    control_score
)

display({
    "greedy": greedy_dr,
    "optimized": optimized_dr
})

{'greedy': {'incremental_value_per_customer': 3.2141473946385397,
  'ci_low': 1.7989964825346325,
  'ci_high': 4.825474064734794},
 'optimized': {'incremental_value_per_customer': 3.593115951153107,
  'ci_low': 2.0192021774667173,
  'ci_high': 5.055255232042429}}

In [35]:
def per_100k(result):
    return {
        "value_per_100k":
            result[
                "incremental_value_per_customer"
            ] * 100_000,
        "ci_low_per_100k":
            result["ci_low"] * 100_000,
        "ci_high_per_100k":
            result["ci_high"] * 100_000
    }

display({
    "greedy": per_100k(greedy_dr),
    "optimized": per_100k(optimized_dr)
})

{'greedy': {'value_per_100k': 321414.739463854,
  'ci_low_per_100k': 179899.64825346324,
  'ci_high_per_100k': 482547.4064734794},
 'optimized': {'value_per_100k': 359311.5951153107,
  'ci_low_per_100k': 201920.21774667173,
  'ci_high_per_100k': 505525.52320424287}}

# Parte M — salvar

In [36]:
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

scored.to_parquet(
    RESULTS_DIR
    / "card_oot_scored_decisioning.parquet",
    index=False
)

greedy_policy.to_csv(
    RESULTS_DIR / "greedy_policy.csv",
    index=False
)

optimized_policy.to_csv(
    RESULTS_DIR / "optimized_policy.csv",
    index=False
)

predicted_comparison.to_csv(
    RESULTS_DIR / "policy_comparison.csv",
    index=False
)

print("POLÍTICAS SALVAS")

POLÍTICAS SALVAS
